In [1]:
import pandas as pd
import numpy as np
import json

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = np.load('../data/processed/y_train.npy')
y_test = np.load('../data/processed/y_test.npy')

with open('../data/processed/label_mapping.json') as f:
    label_mapping = json.load(f)

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
X_test_scaled = scaler.transform(X_test.values)

In [3]:
X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_reshaped = X_test_scaled.reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

In [4]:
import tensorflow as tf
y_train_onehot = tf.keras.utils.to_categorical(y_train)
y_test_onehot = tf.keras.utils.to_categorical(y_test)

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

num_classes = y_train_onehot.shape[1]
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(23, 1)),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

C:\Users\LOQ\anaconda3\envs\traffic-classifier\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,          # stop if val_loss doesn't improve for 5 epochs in a row
    restore_best_weights=True  # roll back to the best-performing epoch, not just the last one
)

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(
    X_train_reshaped, y_train_onehot,
    epochs=100,           # upper limit — early stopping will likely cut it short
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.3152 - loss: 2.0218 - val_accuracy: 0.4063 - val_loss: 1.7386
Epoch 2/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.3875 - loss: 1.7518 - val_accuracy: 0.4194 - val_loss: 1.6279
Epoch 3/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4199 - loss: 1.6485 - val_accuracy: 0.4699 - val_loss: 1.5337
Epoch 4/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4484 - loss: 1.5698 - val_accuracy: 0.4905 - val_loss: 1.4644
Epoch 5/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4666 - loss: 1.5166 - val_accuracy: 0.4768 - val_loss: 1.4328
Epoch 6/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4827 - loss: 1.4698 - val_accuracy: 0.5254 - val_loss: 1.3777
Epoch 7/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.4950 - loss: 1.4363 - val_accuracy: 0.5227 - val_loss: 1.3622
Epoch 8/100
1195/1195 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5035 - loss: 1

In [7]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

y_pred_probs = model.predict(X_test_reshaped)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

374/374 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Test Accuracy: 0.6298777424217049
              precision    recall  f1-score   support

           0       0.61      0.83      0.70      2000
           1       0.54      0.42      0.47       501
           2       0.73      0.54      0.62       795
           3       0.49      0.14      0.22       273
           4       0.57      0.73      0.64       800
           5       0.79      0.29      0.42       257
           6       0.74      0.92      0.82      1297
           7       0.60      0.59      0.60      2000
           8       0.55      0.30      0.38       568
           9       0.70      0.45      0.55       941
          10       0.57      0.77      0.65       489
          11       0.50      0.65      0.57       683
          12       0.68      0.70      0.69       223
          13       0.75      0.52      0.62      1115

    accuracy                           0.63     11942
   macro avg       0.63      0.56      0.57     11942
weigh

In [8]:
model.save('../models/cnn_model.h5')
print("Model saved successfully.")

Model saved successfully.


**Note:** CNN accuracy plateaued at ~63% despite hyperparameter tuning (Dropout, EarlyStopping), consistent with the expectation that Conv1D's assumption of local/spatial feature correlation does not hold for independent flow-level statistical features.